# Modelo A4 — Perfil + popularidad + contenido de productos

## Hipótesis

Los resultados del Modelo A2 mostraron que combinar la información de usuarios con perfiles similares con la popularidad global de los productos mejora considerablemente el rendimiento del recomendador.

Sin embargo, A2 no utiliza información sobre las características propias de los productos.

En esta versión se incorpora una tercera señal basada en el contenido de los productos. Para representarlos se utilizan las características de producto previamente transformadas mediante TF-IDF.

El objetivo es analizar si la combinación de la información del perfil del usuario, la popularidad global y la similitud de contenido permite mejorar el ranking de las recomendaciones.

## Método

El modelo sigue los siguientes pasos:

1. Codifica las cuatro características del usuario:
   - tipo de piel
   - tono de piel
   - color de ojos
   - color de cabello

2. Calcula la similitud coseno entre el perfil del usuario y los perfiles disponibles en entrenamiento.

3. Selecciona los K usuarios más similares.

4. Busca los productos con interacciones positivas de esos vecinos y calcula un puntaje ponderado según la similitud de cada vecino.

5. Calcula la popularidad global de los productos utilizando únicamente las interacciones positivas del conjunto de entrenamiento.

6. Utiliza la representación TF-IDF de los productos para incorporar información relacionada con su contenido.

7. Combina las señales obtenidas para calcular el puntaje final de cada producto.

8. Ordena los productos según el puntaje final y genera el Top-10.

## Ejemplo conceptual

Supongamos un usuario con piel grasa, tono oscuro, ojos azules y cabello castaño.

Primero, el modelo busca usuarios con perfiles similares y analiza qué productos tuvieron interacciones positivas entre esos vecinos.

Luego incorpora la popularidad general de los productos, como se hacía en A2.

Finalmente, utiliza la información de contenido de los productos para identificar productos relacionados con aquellos que fueron valorados positivamente por los usuarios similares.

De esta manera, A4 combina tres tipos de información:

- características del usuario;
- comportamiento general de los productos;
- características de contenido de los productos.

El objetivo es incorporar una señal adicional y diferente de las utilizadas anteriormente, manteniendo el escenario de recomendación para usuarios nuevos sin historial.

## 1. Librerías, datos y división de los datos

En esta primera parte se importan las librerías necesarias y se carga `sephora_common.py`, que contiene las funciones compartidas para la preparación, división y evaluación de los datos.

Para mantener la misma comparación entre los modelos, se utilizan:

- 500 usuarios para validación.
- 500 usuarios para test.
- Semilla aleatoria = 42.

Los datos principales utilizados provienen de `data/processed/`, principalmente de `reviews_clean.parquet` y `products_clean.parquet`.

Además, en esta versión se utilizan las características de contenido de los productos generadas previamente mediante TF-IDF y almacenadas en `data/features/`.

Estas características permiten representar numéricamente la información de los productos y serán utilizadas como una señal adicional dentro del recomendador.

La división de entrenamiento, validación y test se mantiene igual que en los modelos anteriores para poder comparar los resultados bajo las mismas condiciones.

In [1]:
from pathlib import Path
import sys
from collections import Counter

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display


# Buscamos la carpeta principal del proyecto
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/processed"
    )


# Buscamos el archivo con las funciones comunes
helper_dir = next(
    (
        p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
        if (p / "sephora_common.py").exists()
    ),
    None
)

if helper_dir is None:
    raise FileNotFoundError(
        "No se encontró sephora_common.py"
    )


# Agregamos la carpeta para poder importar el archivo
sys.path.insert(0, str(helper_dir))

import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores


# Cantidad de usuarios que usamos para evaluar
N_VALIDACION = 500
N_TEST = 500

# Semilla para obtener siempre la misma división de datos
SEMILLA = 42


# Cargamos los datos y dejamos preparado el contexto del experimento
ctx = sc.load_context(
    N_VALIDACION,
    N_TEST,
    SEMILLA
)


print("Protocolo:", ctx.protocol_id[:12])
print("Resultados:", ctx.output_dir)


# Mostramos la información general del experimento
datos_manifest = {
    k: v
    for k, v in ctx.manifest.items()
    if k not in ["validation_cases", "test_cases", "versions"]
}

display(
    pd.Series(datos_manifest).to_frame("valor")
)

display(
    pd.DataFrame(
        ctx.manifest["versions"],
        index=[0]
    )
)

Protocolo: 99fc64b0ea63
Resultados: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63


,valor
version,cold-start-users-v2.0
seed,42
protocol_id,99fc64b0ea63f0e1e20f6a8a2d96aee1222e401c635a55...
reviews_input,1093855
reviews_valid_before_pair_dedup,1093855
reviews_unique_pairs,1088886
removed_duplicate_pairs,4969
catalog_skincare,2420
users_total,503216
eligible_users,161396


,python,numpy,pandas,scipy,scikit_learn
0,3.14.5,2.4.6,3.0.3,1.18.1,1.9.0


## 2. Condiciones comunes de comparación

Para poder comparar los modelos de forma justa, todos utilizan las mismas condiciones de evaluación.

- **Datos del usuario:** tipo de piel, tono de piel, color de ojos y color de cabello. Cada modelo puede utilizar estas características de manera diferente.
- **Productos:** se trabaja únicamente con productos de la categoría Skincare.
- **Interacción positiva:** se considera positiva cuando el usuario dio un `rating >= 4` o cuando `is_recommended = True`.
- **Validación:** se utilizan 500 usuarios para probar las distintas configuraciones y elegir la mejor.
- **Test:** se utilizan otros 500 usuarios para evaluar la configuración seleccionada.
- **Producto objetivo:** para cada usuario se oculta un producto positivo y se evalúa si el modelo logra recomendarlo dentro del Top-10.
- **Entrenamiento:** los usuarios utilizados para validación y test no participan del conjunto de entrenamiento.

Para la evaluación se seleccionan usuarios que tengan al menos dos interacciones positivas y que cuenten con información de tipo y tono de piel.

Los productos positivos que se utilizan para evaluar al usuario permanecen ocultos para el modelo. De esta manera se busca evitar que el modelo utilice información que no debería conocer al momento de generar la recomendación.

La misma división de datos y la misma semilla se mantienen para todos los modelos, permitiendo comparar sus resultados bajo las mismas condiciones.

## 3. Cómo funciona el algoritmo

El modelo combina tres señales para generar las recomendaciones: la similitud entre perfiles, la popularidad global y el contenido de los productos.

1. **Preparación de los perfiles:** se toman los usuarios del conjunto de entrenamiento y sus cuatro características: tipo de piel, tono de piel, color de ojos y color de cabello. Estas variables se transforman para poder comparar los perfiles.

2. **Búsqueda de vecinos:** para cada usuario se calcula la similitud coseno con los perfiles de entrenamiento y se seleccionan los K usuarios más parecidos.

3. **Puntaje por similitud:** se buscan los productos con interacciones positivas de esos vecinos. Cada producto recibe un puntaje según la similitud del vecino que lo recomendó.

4. **Puntaje por popularidad:** se calcula la popularidad de cada producto utilizando únicamente las interacciones positivas del conjunto de entrenamiento.

5. **Perfil de contenido:** a partir de los productos valorados positivamente por los vecinos se construye una representación de contenido utilizando las características TF-IDF generadas previamente.

6. **Puntaje por contenido:** se calcula la similitud coseno entre ese perfil de contenido y los productos disponibles. De esta manera, los productos con características similares reciben un puntaje mayor.

7. **Combinación de los puntajes:** se combinan las tres señales utilizando dos parámetros:

   - `alpha`: controla el peso del puntaje basado en perfiles.
   - `beta`: controla el peso del contenido.
   - El peso restante corresponde a la popularidad global.

   Los tres pesos suman 1.

8. **Generación del Top-10:** los productos se ordenan según el puntaje final y se seleccionan los 10 primeros.

### Diferencia respecto al Modelo A2

El Modelo A2 utilizaba únicamente la información de los usuarios similares y la popularidad global de los productos.

A4 mantiene esas dos señales e incorpora información sobre el contenido de los productos mediante TF-IDF. El objetivo es analizar si esta nueva información permite identificar productos relacionados con las preferencias observadas entre usuarios con perfiles similares y mejorar su posición dentro del ranking.

### Limitación

El modelo continúa utilizando solamente cuatro características para representar al usuario. Además, la señal de contenido depende de la información disponible en los productos y de la representación obtenida mediante TF-IDF, por lo que productos con descripciones similares no necesariamente representan exactamente las mismas preferencias de los usuarios.

In [2]:
# -------------------------------------------------
# Cargamos las características de contenido
# de los productos generadas en Feature Engineering
# -------------------------------------------------

FEATURES_DIR = PROJECT_ROOT / "data" / "features"

TFIDF_PATH = FEATURES_DIR / "products_tfidf.npz"
PRODUCT_INDEX_PATH = FEATURES_DIR / "products_id_index.parquet"


# Verificamos que los archivos existan
if not TFIDF_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró: {TFIDF_PATH}"
    )

if not PRODUCT_INDEX_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró: {PRODUCT_INDEX_PATH}"
    )


# Cargamos la matriz TF-IDF
X_products_tfidf = sparse.load_npz(
    TFIDF_PATH
)


# Cargamos la relación entre filas y productos
products_id_index = pd.read_parquet(
    PRODUCT_INDEX_PATH
)


print(
    "Dimensiones TF-IDF:",
    X_products_tfidf.shape
)

print(
    "Cantidad de productos en el índice:",
    len(products_id_index)
)

display(
    products_id_index.head()
)

Dimensiones TF-IDF: (8494, 500)
Cantidad de productos en el índice: 8494


,product_id
0,P473671
1,P473668
2,P473662
3,P473660
4,P473658


In [3]:
# Verificamos que cada fila de la matriz
# corresponda a un producto del índice
if X_products_tfidf.shape[0] != len(products_id_index):
    raise ValueError(
        "La cantidad de filas de TF-IDF no coincide "
        "con la cantidad de productos del índice"
    )

print("Matriz TF-IDF e índice de productos: OK")

Matriz TF-IDF e índice de productos: OK


In [ ]:
class ProfilePopularityContent:
    """
    Recomendador que combina tres señales:

    1. Productos valorados positivamente por usuarios con perfiles similares.
    2. Popularidad de los productos dentro del conjunto de entrenamiento.
    3. Similitud de contenido entre productos mediante TF-IDF.
    """

    def __init__(
        self,
        ctx,
        X_products_tfidf,
        products_id_index
    ):

        # Usuarios que pertenecen al conjunto de entrenamiento
        self.profiles = ctx.profiles.loc[
            sorted(ctx.train.author_id.unique())
        ]


        # -------------------------------------------------
        # 1. Preparación de los perfiles de usuarios
        # -------------------------------------------------

        self.encoder = DictVectorizer(
            sparse=True
        )

        records = self.profiles.to_dict(
            "records"
        )

        self.encoder.fit(
            records
            + [{c: "Unknown" for c in PROFILE_COLS}]
        )


        # Valores permitidos para cada característica
        self.allowed = {
            c: set(self.profiles[c]) | {"Unknown"}
            for c in PROFILE_COLS
        }


        # Agrupamos usuarios que tienen exactamente
        # el mismo perfil
        self.groups = {}

        for uid, row in self.profiles.iterrows():

            perfil = tuple(
                row[c]
                for c in PROFILE_COLS
            )

            self.groups.setdefault(
                perfil,
                []
            ).append(uid)


        # Guardamos los perfiles distintos
        self.signatures = list(
            self.groups
        )


        # Transformamos los perfiles
        # para calcular similitud
        self.X = self.encoder.transform([
            dict(zip(PROFILE_COLS, perfil))
            for perfil in self.signatures
        ])


        # Productos positivos de cada usuario
        self.positives = (
            ctx.train[ctx.train.positive]
            .groupby("author_id")
            .product_id
            .apply(list)
            .to_dict()
        )


        # -------------------------------------------------
        # 2. Popularidad global
        # -------------------------------------------------

        popularidad = (
            ctx.train[ctx.train.positive]
            .groupby("product_id")
            .size()
            .reset_index(name="popularity")
        )


        # Normalizamos entre 0 y 1
        max_popularity = popularidad[
            "popularity"
        ].max()

        popularidad["popularity_score"] = (
            popularidad["popularity"]
            / max_popularity
        )


        self.popularity = dict(
            zip(
                popularidad["product_id"],
                popularidad["popularity_score"]
            )
        )


        # -------------------------------------------------
        # 3. Características TF-IDF de los productos
        # -------------------------------------------------

        self.X_products_tfidf = (
            X_products_tfidf
        )


        # Productos que pertenecen al catálogo
        # utilizado en este experimento
        productos_catalogo = set(
            ctx.catalog["product_id"]
        )


        # Relacionamos cada product_id con su fila
        # dentro de la matriz TF-IDF.
        # Conservamos solamente productos del catálogo.
        self.product_to_row = {
            product_id: fila
            for fila, product_id
            in enumerate(
                products_id_index["product_id"]
            )
            if product_id in productos_catalogo
        }


        self.n_train_users = len(
            self.profiles
        )


    def neighbors(
        self,
        profile,
        k
    ):

        if not 1 <= k <= self.n_train_users:
            raise ValueError(
                f"K={k} inválido para "
                f"{self.n_train_users} usuarios de entrenamiento"
            )


        # Armamos el perfil del usuario
        key = tuple(
            profile.get(c, "Unknown")
            if profile.get(c, "Unknown")
            in self.allowed[c]
            else "Unknown"
            for c in PROFILE_COLS
        )


        # Transformamos el perfil
        query = self.encoder.transform([
            dict(zip(PROFILE_COLS, key))
        ])


        # Calculamos la similitud coseno
        similarities = cosine_similarity(
            query,
            self.X
        ).ravel()


        vecinos = []


        # Ordenamos los perfiles
        # desde el más parecido
        orden = np.argsort(
            -similarities
        )


        for indice in orden:

            similitud = similarities[
                indice
            ]

            usuarios_grupo = self.groups[
                self.signatures[indice]
            ]


            for uid in usuarios_grupo:

                vecinos.append(
                    (uid, similitud)
                )

                if len(vecinos) == k:
                    return vecinos


        return vecinos


    def recommend(
        self,
        profile,
        config,
        top_n=10,
        exclude=()
    ):

        k = int(
            config["k"]
        )

        alpha = float(
            config["alpha"]
        )

        beta = float(
            config["beta"]
        )


        # -------------------------------------------------
        # 1. Puntaje según perfiles similares
        # -------------------------------------------------

        profile_scores = Counter()

        vecinos = self.neighbors(
            profile,
            k
        )


        for uid, similitud in vecinos:

            productos_usuario = (
                self.positives.get(
                    uid,
                    []
                )
            )

            for product_id in productos_usuario:

                profile_scores[
                    product_id
                ] += similitud


        # Normalizamos el puntaje de perfil
        if profile_scores:

            max_profile_score = max(
                profile_scores.values()
            )

        else:

            max_profile_score = 1


        # -------------------------------------------------
        # 2. Construimos el perfil de contenido
        # -------------------------------------------------

        filas_productos = []

        pesos_productos = []


        for product_id, score in profile_scores.items():

            if product_id in self.product_to_row:

                filas_productos.append(
                    self.product_to_row[
                        product_id
                    ]
                )

                pesos_productos.append(
                    score
                )


        # -------------------------------------------------
        # 3. Calculamos similitud de contenido
        # -------------------------------------------------

        content_scores = {}


        if filas_productos:

            matriz_productos = (
                self.X_products_tfidf[
                    filas_productos
                ]
            )

            pesos = np.array(
                pesos_productos
            )

            # Construimos un perfil promedio
            # de contenido de los productos
            perfil_contenido = (
                matriz_productos
                .multiply(
                    pesos[:, None]
                )
                .sum(axis=0)
            )


            # Convertimos el resultado a un array de NumPy
            # para poder utilizar cosine_similarity
            perfil_contenido = np.asarray(
                perfil_contenido
            )


            similitudes_contenido = (
                cosine_similarity(
                    perfil_contenido,
                    self.X_products_tfidf
                ).ravel()
            )


            for product_id, fila in (
                self.product_to_row.items()
            ):

                content_scores[
                    product_id
                ] = (
                    similitudes_contenido[
                        fila
                    ]
                )


        # -------------------------------------------------
        # 4. Combinamos las tres señales
        # -------------------------------------------------

        productos = (
            set(profile_scores.keys())
            | set(self.popularity.keys())
            | set(content_scores.keys())
        )


        resultados = []


        for product_id in productos:

            score_perfil = (
                profile_scores.get(
                    product_id,
                    0
                )
                / max_profile_score
            )

            score_popularidad = (
                self.popularity.get(
                    product_id,
                    0
                )
            )

            score_contenido = (
                content_scores.get(
                    product_id,
                    0
                )
            )


            # Alpha = peso del perfil
            # Beta = peso del contenido
            # El peso restante corresponde
            # a la popularidad
            peso_popularidad = (
                1 - alpha - beta
            )


            score_final = (
                alpha * score_perfil
                + peso_popularidad * score_popularidad
                + beta * score_contenido
            )


            resultados.append({
                "product_id": product_id,
                "score": score_final
            })


        frame = pd.DataFrame(
            resultados
        )


        # Ordenamos y devolvemos el Top-10
        return rank_scores(
            frame,
            "score",
            top_n,
            exclude
        )

## 4. Preparación del modelo y configuraciones

In [12]:
# Creamos el modelo que combina
# perfil, popularidad y contenido
model = ProfilePopularityContent(
    ctx,
    X_products_tfidf,
    products_id_index
)


print(
    "Usuarios de entrenamiento:",
    model.n_train_users
)

print(
    "Perfiles distintos:",
    len(model.signatures)
)


# Configuraciones que vamos a probar en validación
CONFIGURACIONES = [
    {
        "k": k,
        "alpha": alpha,
        "beta": beta
    }
    for k in [100, 250, 500]
    for alpha in [0.25, 0.50, 0.75]
    for beta in [0.10, 0.20, 0.30]
    if k <= model.n_train_users
    and alpha + beta <= 1
]


if not CONFIGURACIONES:
    raise ValueError(
        "No hay configuraciones válidas para evaluar"
    )


print(
    "Cantidad de configuraciones:",
    len(CONFIGURACIONES)
)

Usuarios de entrenamiento: 502216
Perfiles distintos: 1842
Cantidad de configuraciones: 24


## 5. Ajuste en validación y evaluación final

Para evaluar el modelo se utilizan principalmente las siguientes métricas:

- **Hit Rate@10 (HR@10):** indica en cuántos usuarios el producto que ocultamos para la prueba aparece dentro de las 10 recomendaciones.

- **NDCG@10:** además de considerar si el producto aparece en el Top-10, tiene en cuenta su posición. Cuanto más arriba aparece el producto correcto, mejor es el resultado.

- **Cobertura de usuarios:** indica qué porcentaje de usuarios recibe al menos una recomendación.

- **Top-10 completo:** indica qué porcentaje de usuarios recibe las 10 recomendaciones.

- **Cobertura de catálogo:** indica qué proporción de los productos disponibles aparece al menos una vez entre las recomendaciones generadas.

En esta versión se prueban distintas combinaciones de tres parámetros:

- **K:** cantidad de usuarios vecinos considerados.
- **alpha:** peso asignado a la señal basada en perfiles similares.
- **beta:** peso asignado a la señal basada en el contenido de los productos.

El peso restante se asigna a la popularidad global de los productos.

Por lo tanto:

- Un `alpha` más alto aumenta la importancia de la información obtenida de los usuarios similares.
- Un `beta` más alto aumenta la importancia de la similitud de contenido entre productos.
- El peso de la popularidad corresponde al valor restante después de considerar `alpha` y `beta`.

Las distintas combinaciones de K, alpha y beta se evalúan primero utilizando los 500 usuarios de validación.

La mejor configuración se selecciona principalmente según su HR@10 y, en caso de empate, se utiliza NDCG@10.

Una vez elegida la mejor configuración, se evalúa con los 500 usuarios de test.

Finalmente, los resultados del modelo se comparan con el mismo baseline de popularidad utilizado en los modelos anteriores.

In [13]:
mejor_config, validacion, metricas_test, detalle_test, baseline_test = sc.run_experiment(
    model,
    ctx,
    CONFIGURACIONES
)


print("Resultados de validación:")
display(validacion)


print(
    "Configuración elegida:",
    mejor_config
)


print("Resultados finales en test:")
display(
    pd.DataFrame(
        [metricas_test, baseline_test],
        index=[
            "Modelo A4",
            "Baseline global"
        ]
    )
)

Validación: {'k': 100, 'alpha': 0.25, 'beta': 0.1}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 0.25, 'beta': 0.2}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 0.25, 'beta': 0.3}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 0.5, 'beta': 0.1}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 0.5, 'beta': 0.2}
Evaluados 50/500
Evaluados 100/

,config,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
0,"{'k': 100, 'alpha': 0.25, 'beta': 0.1}",500,39,0.078,0.042645,1.0,1.0,0.037190,10.0,0.026000
1,"{'k': 100, 'alpha': 0.25, 'beta': 0.2}",500,37,0.074,0.041384,1.0,1.0,0.045455,10.0,0.036098
2,"{'k': 100, 'alpha': 0.25, 'beta': 0.3}",500,35,0.070,0.039485,1.0,1.0,0.056198,10.0,0.029181
3,"{'k': 100, 'alpha': 0.5, 'beta': 0.1}",500,36,0.072,0.042183,1.0,1.0,0.091322,10.0,0.020868
4,"{'k': 100, 'alpha': 0.5, 'beta': 0.2}",500,35,0.070,0.040827,1.0,1.0,0.101240,10.0,0.020008
5,"{'k': 100, 'alpha': 0.5, 'beta': 0.3}",500,33,0.066,0.038791,1.0,1.0,0.117769,10.0,0.019111
6,"{'k': 100, 'alpha': 0.75, 'beta': 0.1}",500,36,0.072,0.040759,1.0,1.0,0.108678,10.0,0.019628
7,"{'k': 100, 'alpha': 0.75, 'beta': 0.2}",500,30,0.060,0.035020,1.0,1.0,0.136777,10.0,0.021234
8,"{'k': 250, 'alpha': 0.25, 'beta': 0.1}",500,39,0.078,0.041675,1.0,1.0,0.020661,10.0,0.020633
9,"{'k': 250, 'alpha': 0.25, 'beta': 0.2}",500,38,0.076,0.040961,1.0,1.0,0.027273,10.0,0.020013


Configuración elegida: {'k': 500, 'alpha': 0.25, 'beta': 0.1}
Resultados finales en test:


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
Modelo A4,500,36,0.072,0.032986,1.0,1.0,0.015289,10.0,0.021737
Baseline global,500,37,0.074,0.033161,1.0,1.0,0.004132,10.0,0.003719


## 6. Ejemplo de recomendación para un usuario nuevo

Para visualizar cómo funcionaría el modelo en una situación real, se utiliza un perfil de ejemplo y se generan sus 10 productos recomendados.

En este caso no se está evaluando el modelo, sino mostrando el resultado que recibiría un usuario nuevo al ingresar sus características.

El modelo busca usuarios con perfiles similares y obtiene los productos con interacciones positivas de esos vecinos.

A partir de esta información se generan tres señales:

- un puntaje basado en la similitud de los perfiles;
- un puntaje basado en la popularidad global de los productos;
- un puntaje basado en la similitud de contenido de los productos mediante TF-IDF.

Para la señal de contenido, el modelo utiliza los productos valorados positivamente por los usuarios similares para construir un perfil de contenido y buscar productos con características relacionadas.

El peso de las señales depende de los valores de `alpha` y `beta` seleccionados durante la etapa de validación.

Finalmente, los tres puntajes se combinan, los productos se ordenan según el resultado obtenido y se muestran las 10 recomendaciones con mayor puntuación.

In [16]:
# Perfil de ejemplo para visualizar
# las recomendaciones del modelo

PERFIL_EJEMPLO = {
    "skin_type": "oily",
    "skin_tone": "dark",
    "eye_color": "blue",
    "hair_color": "brown"
}

perfil, top10 = sc.example_top10(
    model,
    ctx,
    mejor_config,
    PERFIL_EJEMPLO
)

print("Perfil utilizado:")
display(
    pd.DataFrame([perfil])
)

print("Top-10 recomendado:")
display(
    top10
)

print(
    "Productos devueltos:",
    len(top10)
)

Perfil utilizado:


,skin_type,skin_tone,eye_color,hair_color
0,oily,dark,blue,brown


Top-10 recomendado:


,position,product_id,product_name,brand_name,secondary_category,price_usd
0,1,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Lip Balms & Treatments,24.0
1,2,P7880,Soy Hydrating Gentle Face Cleanser,fresh,Cleansers,39.0
2,3,P269122,Alpha Beta Extra Strength Daily Peel Pads,Dr. Dennis Gross Skincare,Treatments,92.0
3,4,P248407,Ultra Repair Cream Intense Hydration,First Aid Beauty,Moisturizers,38.0
4,5,P427417,Niacinamide 10% + Zinc 1% Oil Control Serum,The Ordinary,Treatments,6.0
5,6,P394639,The True Cream Aqua Bomb,belif,Moisturizers,38.0
6,7,P417238,Green Clean Makeup Removing Cleansing Balm,Farmacy,Cleansers,36.0
7,8,P411540,Cicapair Tiger Grass Color Correcting Treatmen...,Dr. Jart+,Moisturizers,52.0
8,9,P309308,Good Genes All-In-One AHA Lactic Acid Treatment,Sunday Riley,Treatments,122.0
9,10,P471546,Mini Cicapair Tiger Grass Color Correcting Tre...,Dr. Jart+,Mini Size,21.0


Productos devueltos: 10


## 7. Cierre y resultados

El Modelo A4 incorporó información sobre el contenido de los productos mediante TF-IDF a las señales de perfil y popularidad utilizadas anteriormente.

Durante la etapa de validación, la configuración seleccionada fue:

- K = 500 vecinos.
- alpha = 0.25.
- beta = 0.10.

Esto significa que el puntaje final asigna un 25 % de peso a la señal basada en perfiles similares, un 10 % a la similitud de contenido y el 65 % restante a la popularidad global.

En el conjunto de test, el modelo obtuvo:

- HR@10 = 7,2 %, con 36 aciertos sobre 500 usuarios.
- NDCG@10 = 0,0330.

El baseline de popularidad obtuvo:

- HR@10 = 7,4 %, con 37 aciertos sobre 500 usuarios.
- NDCG@10 = 0,0332.

Por lo tanto, A4 quedó a un solo acierto del baseline en HR@10 y obtuvo un NDCG@10 ligeramente inferior.

En comparación con A2, ambos modelos alcanzaron un HR@10 de 7,2 %, pero A2 obtuvo un NDCG@10 superior (0,0342 frente a 0,0330).

Los resultados indican que incorporar la similitud de contenido mediante TF-IDF no produjo una mejora adicional sobre A2. Por este motivo, A4 se conserva como un experimento evaluado y descartado, mientras que A2 continúa siendo la mejor versión de la familia A.  

In [15]:
sc.conclusion(
    metricas_test,
    baseline_test
)


extra = {
    "algorithm": "ProfilePopularityContent",
    "version": "perfil_popularidad_contenido"
}


archivo = sc.save_result(
    ctx,
    "01_A4_perfil_popularidad_contenido",
    mejor_config,
    validacion,
    metricas_test,
    detalle_test,
    baseline_test,
    top10,
    extra
)


print(
    "Resultado guardado:",
    archivo
)

HR@10 test: 7.20%; baseline: 7.40%; diferencia: -0.20 puntos porcentuales.
Cobertura de usuarios: 100.00%; catálogo: 1.53%.
En esta muestra no supera al baseline en HR. No declararlo ganador por personalización solamente.
Un hit recupera una preferencia observada; no prueba eficacia ni compatibilidad dermatológica.
Resultado guardado: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63\01_A4_perfil_popularidad_contenido.json
